# Redes Neuronales Densas — Modelos de Forecasting

Objetivo: Entrenar modelos de redes neuronales con capas densas para predecir el promedio del log-return de cierre futuro de 23 activos del S&P 500, a través de 16 combinaciones de ventanas temporales de entrada y salida

Métrica: MAE

In [ ]:
import sys
sys.path.append('..')
os.chdir('..')

## 1. Importar y configuración

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, History

# Infraestructura compartida del proyecto
from src.data_pipeline import load_data, get_train_test
from src.evaluation import compute_mae, save_results, count_parameters
from src.plotting import plot_training_curves
from config import INPUT_WINDOWS, OUTPUT_WINDOWS, FIGURES_DIR, MODELS_DIR

import os
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.21.0


## 2. Carga de datos

In [11]:
returns = load_data()
print(f"\nShape: {returns.shape}")
print(f"Columnas: {list(returns.columns)}")
returns.head()

[*********************100%***********************]  23 of 23 completed


Datos cargados: 16195 días, 23 activos
Rango: 1962-01-03 → 2026-05-08

Shape: (16195, 23)
Columnas: ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 'MSI', 'PG', 'XOM']


Ticker,AEP,BA,CAT,CNP,CVX,DIS,DTE,ED,GD,GE,...,IP,JNJ,KO,KR,MMM,MO,MRK,MSI,PG,XOM
Date,,,,,,,,,,,,,,,,,,,,,
1962-01-03,-0.001823,0.019802,0.009693,-0.009820,-0.002260,0.013333,-0.008264,0.000000,0.032925,-0.010084,...,-0.013745,-0.015671,-0.022532,0.020790,0.007491,-0.018517,-0.042802,-0.002911,-0.010990,0.014744
1962-01-04,-0.014706,-0.009852,0.025398,0.000000,-0.009092,0.000000,-0.008334,-0.003091,0.004040,-0.011895,...,-0.003467,-0.010578,0.007569,-0.004124,0.000000,0.006984,-0.010256,-0.008784,-0.016713,0.002435
1962-01-05,-0.022472,-0.020001,0.009360,-0.024419,-0.025435,0.003308,-0.021142,-0.021910,0.004024,-0.025976,...,0.010364,-0.016090,-0.022876,-0.025107,-0.026467,0.011533,-0.034461,-0.005900,-0.007048,-0.022140
1962-01-08,-0.007604,0.002522,0.006193,-0.011300,-0.004694,-0.003308,0.002134,0.004736,0.015938,-0.001756,...,-0.020835,-0.016347,-0.010335,-0.004245,-0.005763,-0.006903,0.003043,-0.017911,-0.027241,-0.002492
1962-01-09,-0.007664,0.002515,0.009216,0.000000,0.014019,0.019677,0.002131,-0.001577,0.003944,0.005258,...,-0.014135,0.010929,0.018017,-0.021506,0.000000,-0.018648,0.004550,-0.024391,-0.001455,-0.002497
